In [4]:
!pip install transformers torch sentencepiece --quiet
!pip install git+https://github.com/csebuetnlp/normalizer

  Cloning https://github.com/csebuetnlp/normalizer to /tmp/pip-req-build-krk115hb
  Running command git clone --filter=blob:none --quiet https://github.com/csebuetnlp/normalizer /tmp/pip-req-build-krk115hb
  Resolved https://github.com/csebuetnlp/normalizer to commit d405944dde5ceeacb7c2fd3245ae2a9dea5f35c9
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for normalizer: filename=normalizer-0.0.1-py3-none-any.whl size=6860 sha256=48c87860e7c5f2bcb5aafcacbd76c87aa6d48369b0ddd3b63f45c2ea49879f4e
  Stored in directory: /tmp/pip-ephem-wheel-cache-vytuy2kb/wheels/f9/d8/55/a13fa77440d3e80bf10ff80176ba67c7a0543a67827ef0b8eb
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186456 sha256=996cdc0213c33ba54cc10d867565dbfbd3426cb0

In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from normalizer import normalize

In [6]:

models = ["Showrov5843/banglat5-dialect-to-standard", "csebuetnlp/banglat5", "csebuetnlp/banglat5_nmt_bn_en"]
model_name = models[2]

do_normalize = True
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
2026-01-08 04:26:00.181795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767846360.446664      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767846360.522840      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [7]:
def banglat5_inference(text, max_len=100):
    txt = text
    if do_normalize:
        txt = normalize(text)
    # Tokenize
    inputs = tokenizer(txt, return_tensors="pt", padding=True, truncation=True)
    
    # Generate output tokens
    outputs = model.generate(
        **inputs,
        max_length=max_len,
        num_beams=4,
        early_stopping=True
    )
    
    # Decode and return string
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return decoded[0]

In [8]:
# Example inputs — change these to whatever text you want
# input_texts = [
#     # "মাইআউগ্গা লাল রুঙ্গা শাড়ি পইর্রা মর লগে দেহা হর্তে আইসিলো",   # Bangla text (example)
#     "কাজ: অনুবাদ  উৎস ভাষা: বাংলা লক্ষ্য ভাষা: ইংরেজি  বাক্য: তুমি কি মোরে চেনো?"  # Example if using translation model
# ]

prompts = ["",
         "কাজ: অনুবাদ  উৎস ভাষা: বাংলা লক্ষ্য ভাষা: ইংরেজি  বাক্য: ",
         "Task: Translation Source language: English Target language: Bangla Sentence: "]

prompt = prompts[1]

barishal_texts = [
    ("আসো কোরোহম?","How are you?" ),
    ("এই থাডা পরা গরমে মোর কিস্সু ভাল্লাগেনা", "I don't like anything this summer"),
    ("পলাউগ্গা এউক্কা ধলা রং এর এউক্কা গুন্জি পইর্রা আইসেল" "The boy came wearing a white shirt"),
    ("মাইআউগ্গা শ্যলেত দিআ আঔআ পলাউগ্গারে আক্সের ভালপায়", "The girl loves this boy from Sylhet very much"),
    ("হ্যায় হ্যার গেরামের এউক্কা ইসকুলে পড়াল্যাহা করে", "He studies in a school in his village"),
    ("পোলাডা হ্যার গেরামের ছোডো এউক্কা ইসকুল হইতে পড়াল্যাহা কইররা মেটটিক পাশ করছে", "The boy studied from a small school in his village and passed SSC")
    
]

chittagong_texts = [
    ("বাংলাদেশত ৬৪ ইয়ান জেলা", "64 districts in Bangladesh"),
    ("তোইয়ার হতা বলার ধরণ বহুত সুন্দর", "Your way of speaking is very nice"),
    ("আম দি বানাইত্লেতারিবানা মু দিয়েরে ফারিবানা শরবন", "You can make syrup with mango"),
    ("বুঝি উঠিন্ন ফারির কেন গইজ্জুম", "I could not understand what to do"),
    ("তোঁয়াত্তুন ফরালেহা গইরতো গম ন লাগে", "Do you not like to study?")
]

mymensingh_texts = [
    ("আমরা বেহেই কালকে বাইরে গেসিলাম", "We all went out yesterday"),
    ("কহনো ভাইব্বা দেখছ আমি না থাকলে তোমার কিতা হইতো?", "Have you ever wondered what would happen to you without me?"),
    ("বুইজ্ঝা উঠতে পারতাসিলাম না কীতা করবাম", "I could not understand what to do")
]

noakhali_texts = [
    ("কোন সময় চিন্তা করি সাইসেন আই নো থাইকলে তোওয়ার কিয়া অইত", "Have you ever wondered what would happen to you without me?"),
    ("আন্ডা বেজ্ঞুন কাইল্লা বাইরে গেসিলাম", "We all went out yesterday"),
    ("আমি চলি যাইর তোওয়ার আর কোন দিন ফোন দি কইতান্নো এক্কেনা সময় দো", "I'm leaving, I won't call you any other day, give me some time")
]

sylhet_texts = [
    ("আমরা হকল গতকাইল বাহিরো গিয়েছিলাম", "We all went out yesterday"),
    ("আফনে কিতা অউ বইটা পড়তা চাইন নি?","Do you want to read this book?"),
    ("বাক্কা রাইত অই গেছে এখন গুমাই যাও", "It's late night now go to sleep")
]

standard_bangla_texts = [
    ("ছেলেটি তাঁর গ্রামের ছোটো একটি বিদ্যালয় থেকে পড়ালেখা করে এসএসসি পাশ করেছে", "The boy studied from a small school in his village and passed SSC"),
    ("আম্মু ঐদিন ঠিক কথাই বলেছিলও", "Mom said the right thing that day"),
    ("তুমি কি করো ?", "what are you doing?"),
    ("আমি চলে যাচ্ছি তোমাকে আর কোন দিন ফোন দিয়ে বলবোনা একটু সময় দাও", "I'm leaving, I won't call you any other day, give me some time"),
    ("আমকে আমাদের দেশে ‘ফলের রাজা’ বলা হয়", "Mango is called the 'king of fruits' in our country" )
    
]
# input_texts = [
#     # "মাইআউগ্গা লাল রুঙ্গা শাড়ি পইর্রা মর লগে দেহা হর্তে আইসিলো",   # Bangla text (example)
#     "কাজ: অনুবাদ  উৎস ভাষা: বাংলা লক্ষ্য ভাষা: ইংরেজি  বাক্য: আমনের কি পড়াল্যাহা করতে এক্কালেই ভালো লাগে না?"  # Example if using translation model
# ]


In [ ]:
#Inference
#Barishal
for txt in barishal_texts:
    output = banglat5_inference(txt[0])
    print("Input :", txt[0])
    print("Output:", output)
    print("Actual output:", txt[1])
    print("----")

In [ ]:
#Chittagong

for txt in chittagong_texts:
    output = banglat5_inference(txt[0])
    print("Input :", txt[0])
    print("Output:", output)
    print("Actual output:", txt[1])
    print("----")

In [ ]:
#Mymensingh

for txt in mymensingh_texts:
    output = banglat5_inference(txt[0])
    print("Input :", txt[0])
    print("Output:", output)
    print("Actual output:", txt[1])
    print("----")

In [ ]:
#Noakhali

for txt in noakhali_texts:
    output = banglat5_inference(txt[0])
    print("Input :", txt[0])
    print("Output:", output)
    print("Actual output:", txt[1])
    print("----")

In [ ]:
#Sylhet

for txt in sylhet_texts:
    output = banglat5_inference(txt[0])
    print("Input :", txt[0])
    print("Output:", output)
    print("Actual output:", txt[1])
    print("----")

In [10]:
#Standard Bangla

for txt in standard_bangla_texts:
    output = banglat5_inference(txt[0])
    print("Input :", txt[0])
    print("Output:", output)
    print("Actual output:", txt[1])
    print("----")

Input : ছেলেটি তাঁর গ্রামের ছোটো একটি বিদ্যালয় থেকে পড়ালেখা করে এসএসসি পাশ করেছে
Output: The boy passed SSC from a small school in his village
Actual output: The boy studied from a small school in his village and passed SSC
----
Input : আম্মু ঐদিন ঠিক কথাই বলেছিলও
Output: Mommy was right that day.
Actual output: Mom said the right thing that day
----
Input : তুমি কি করো ?
Output: What do you do?
Actual output: what are you doing?
----
Input : আমি চলে যাচ্ছি তোমাকে আর কোন দিন ফোন দিয়ে বলবোনা একটু সময় দাও
Output: I'm leaving, and I'll never call you again and ask you to give me a moment.
Actual output: I'm leaving, I won't call you any other day, give me some time
----
Input : আমকে আমাদের দেশে ‘ফলের রাজা’ বলা হয়
Output: I am called 'the King of Fruits' in our country
Actual output: Mango is called the 'king of fruits' in our country
----


In [ ]:
#English generation

output = banglat5_inference("Write in english")
print("Output:", output )

In [9]:
#English generation

output = banglat5_inference("ইংরেজিতে লিখুন")
print("Output:", output )

Output: Write in English


## BanglaT5: Just keeps repeating the part of the input given to it


## BanglaT5_bn_en: Can translate some sentences, but starts generating english text for the bangla pronunciation for a lot of sentences

## BanglaT5_dialect_to_standard : cannot generate english text at all, wrong translations in most cases